In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("NullHandling").getOrCreate()

In [2]:
%%writefile bookings.csv

booking_id,customer_name,city,service_type,provider,booking_amount,booking_status,payment_mode
1001,Aarav Mehta,Hyderabad,Flight,IndiGo,6500,Confirmed,UPI
1002,Sana Khan,Bangalore,Hotel,Pearl Grand,4500,Confirmed,Card
1003,John Mathew,,Flight,Air India,12000,Confirmed,UPI
1004,Ayesha Begum,Hyderabad,Hotel,,7500,Pending,Cash
1005,Vikram Rao,Mumbai,Flight,Vistara,,Confirmed,Card
1006,Divya Sharma,Delhi,Flight,IndiGo,5900,Cancelled,
1007,Imran Ali,Pune,Hotel,Budget Inn,2200,,UPI
1008,Meera Nair,Kochi,Hotel,Hill View Resort,7500,Confirmed,Card
1009,Rohan Das,Kolkata,Flight,Air India,7400,Pending,UPI
1010,Nisha Reddy,Bangalore,Flight,British Airways,62000,Confirmed,Card
1011,Farhan Ali,,Hotel,Skyline Suites,22000,Confirmed,
1012,Neha Singh,Hyderabad,,Emirates,28000,Confirmed,UPI
1013,Arjun Verma,Chennai,Flight,,15000,Cancelled,Cash
1014,Kavya Nair,Mumbai,Hotel,Sea View Stay,,Pending,Card
1015,Ravi Kumar,Delhi,Flight,SpiceJet,4800,Confirmed,UPI

Writing bookings.csv


In [3]:
bookings_df = spark.read.csv(
    "bookings.csv",
    header=True,
    inferSchema=True
)

bookings_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [4]:
bookings_df.printSchema()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- provider: string (nullable = true)
 |-- booking_amount: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_mode: string (nullable = true)



In [5]:
bookings_df.count()

15

In [6]:
bookings_df.filter(col("city").isNull()).show()

+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|      1003|  John Mathew|NULL|      Flight|     Air India|         12000|     Confirmed|         UPI|
|      1011|   Farhan Ali|NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NULL|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+



In [7]:
bookings_df.filter(col("provider").isNull()).show()

+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|      1004| Ayesha Begum|Hyderabad|       Hotel|    NULL|          7500|       Pending|        Cash|
|      1013|  Arjun Verma|  Chennai|      Flight|    NULL|         15000|     Cancelled|        Cash|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+



In [8]:
bookings_df.filter(col("booking_amount").isNull()).show()

+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|booking_id|customer_name|  city|service_type|     provider|booking_amount|booking_status|payment_mode|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|      1005|   Vikram Rao|Mumbai|      Flight|      Vistara|          NULL|     Confirmed|        Card|
|      1014|   Kavya Nair|Mumbai|       Hotel|Sea View Stay|          NULL|       Pending|        Card|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+



In [9]:
bookings_df.filter(col("booking_status").isNull()).show()

+----------+-------------+----+------------+----------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|  provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+----------+--------------+--------------+------------+
|      1007|    Imran Ali|Pune|       Hotel|Budget Inn|          2200|          NULL|         UPI|
+----------+-------------+----+------------+----------+--------------+--------------+------------+



In [10]:
bookings_df.filter(col("payment_mode").isNull()).show()

+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name| city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|      1006| Divya Sharma|Delhi|      Flight|        IndiGo|          5900|     Cancelled|        NULL|
|      1011|   Farhan Ali| NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NULL|
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+



In [11]:
bookings_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in bookings_df.columns
]).show()

+----------+-------------+----+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------+--------------+--------------+------------+
|         0|            0|   2|           1|       2|             2|             1|           2|
+----------+-------------+----+------------+--------+--------------+--------------+------------+



In [12]:
bookings_df.na.drop().show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1008|   Meera Nair|    Kochi|       Hotel|Hill View Resort|          7500|     Confirmed|        Card|
|      1009|    Rohan Das|  Kolkata|      Flight|       Air India|          7400|       Pending|         UPI|
|      1010|  Nisha Reddy|Bangalore|      Flight| British Airways|         62000|     Confirmed|        Card|
|      1015|   Ravi Kumar|    Delhi|      Flight|        SpiceJet|          4800|     Confirmed|         UPI|
+---------

In [13]:
bookings_df.na.drop(subset=["booking_amount"]).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      1007|    Imran Ali|     Pune|       Hotel|      Budget Inn|          2200|          NULL|         UPI|
|      100

In [14]:
bookings_df.na.drop(
    subset=["customer_name", "service_type", "booking_amount"]
).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      1007|    Imran Ali|     Pune|       Hotel|      Budget Inn|          2200|          NULL|         UPI|
|      100

In [15]:
bookings_df.na.fill({
    "city": "Unknown"
}).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [16]:
bookings_df.na.fill({
    "provider": "Not Available"
}).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [17]:
bookings_df.na.fill({
    "payment_mode": "Not Provided"
}).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|Not Provided|
|      100

In [18]:
bookings_df.na.fill({
    "booking_status": "Unknown"
}).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [19]:
bookings_df.na.fill({
    "booking_amount": 0
}).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|         UPI|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|        Cash|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|             0|     Confirmed|        Card|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

Clean Df

In [20]:
clean_df = bookings_df.na.fill({
    "city": "Unknown",
    "provider": "Not Available",
    "payment_mode": "Not Provided",
    "booking_status": "Unknown",
    "booking_amount": 0
})

In [21]:
clean_df = clean_df.withColumn(
    "data_quality_status",
    when(
        col("customer_name").isNull() |
        col("service_type").isNull() |
        col("booking_amount").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)

clean_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash|           Complete|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|             0|     Conf

In [22]:
clean_df.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|   14|
|         Incomplete|    1|
+-------------------+-----+



In [23]:
clean_df.filter(
    col("data_quality_status") == "Complete"
).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash|           Complete|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|             0|     Conf

In [24]:
clean_df.filter(
    col("data_quality_status") == "Incomplete"
).show()

+----------+-------------+---------+------------+--------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+-------------------+
|      1012|   Neha Singh|Hyderabad|        NULL|Emirates|         28000|     Confirmed|         UPI|         Incomplete|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+-------------------+



In [25]:
clean_df = clean_df.withColumn(
    "tax",
    col("booking_amount") * 0.05
)

clean_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|   tax|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete| 325.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete| 225.0|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete| 600.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|        Cash|           Complete| 375.0|
|      1005|   Vikram Rao|   Mumbai|     

In [26]:
clean_df = clean_df.withColumn(
    "final_amount",
    col("booking_amount") + col("tax")
)

clean_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|   tax|final_amount|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|         UPI|           Complete| 325.0|      6825.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|        Card|           Complete| 225.0|      4725.0|
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|         UPI|           Complete| 600.0|     12600.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|    

In [27]:
clean_df.filter(
    col("booking_status") == "Confirmed"
).select(
    sum("final_amount").alias("total_revenue")
).show()

+-------------+
|total_revenue|
+-------------+
|     154665.0|
+-------------+



In [28]:
clean_df.na.fill({
    "service_type": "Unknown"
}).groupBy(
    "service_type"
).count().show()

+------------+-----+
|service_type|count|
+------------+-----+
|     Unknown|    1|
|       Hotel|    6|
|      Flight|    8|
+------------+-----+



In [29]:
clean_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    2|
|  Kolkata|    1|
|  Unknown|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [30]:
clean_df.select(
    avg("booking_amount")
).show()

+-------------------+
|avg(booking_amount)|
+-------------------+
| 12353.333333333334|
+-------------------+



In [31]:
bookings_df.na.drop(
    subset=["booking_amount"]
).select(
    avg("booking_amount")
).show()

+-------------------+
|avg(booking_amount)|
+-------------------+
| 14253.846153846154|
+-------------------+



In [32]:
avg_fill_zero = clean_df.select(
    avg("booking_amount")
).collect()[0][0]

avg_drop_null = bookings_df.na.drop(
    subset=["booking_amount"]
).select(
    avg("booking_amount")
).collect()[0][0]

print("Average after filling nulls with 0 :", avg_fill_zero)
print("Average after dropping null rows   :", avg_drop_null)

Average after filling nulls with 0 : 12353.333333333334
Average after dropping null rows   : 14253.846153846154


In [33]:
clean_df.write.mode("overwrite").parquet("clean_bookings.parquet")

JSON File

In [34]:
%%writefile customers.json

[
{
"customer_id": 1,
"name": "Aarav Mehta",
"city": "Hyderabad",
"membership": "Gold",
"contact": {
"phone": "9876500011",
"email": "aarav@mail.com"
},
"preferences": {
"preferred_service": "Flight",
"budget_range": "Medium"
}
},
{
"customer_id": 2,
"name": "Sana Khan",
"city": "Bangalore",
"membership": "Silver",
"contact": {
"phone": null,
"email": "sana@mail.com"
},
"preferences": {
"preferred_service": "Hotel",
"budget_range": null
}
},
{
"customer_id": 3,
"name": "John Mathew",
"city": null,
"membership": "Gold",
"contact": {
"phone": "9876500013",
"email": null
},
"preferences": {
"preferred_service": "Flight",
"budget_range": "High"
}
},
{
"customer_id": 4,
"name": "Ayesha Begum",
"city": "Hyderabad",
"membership": null,
"contact": {
"phone": "9876500014",
"email": "ayesha@mail.com"
},
"preferences": {
"preferred_service": null,
"budget_range": "Low"
}
},
{
"customer_id": 5,
"name": "Vikram Rao",
"city": "Mumbai",
"membership": "Platinum",
"contact": {
"phone": null,
"email": null
},
"preferences": {
"preferred_service": "Flight",
"budget_range": "High"
}
}
]

Writing customers.json


In [35]:
customers_df = spark.read.option(
    "multiline",
    "true"
).json("customers.json")

customers_df.show(truncate=False)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+



In [36]:
customers_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)



In [37]:
customers_df.show(truncate=False)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+



In [38]:
flat_customers_df = customers_df.select(
    "customer_id",
    "name",
    "city",
    "membership",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    col("preferences.preferred_service").alias("preferred_service"),
    col("preferences.budget_range").alias("budget_range")
)

flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [39]:
flat_customers_df.select(
    "name",
    "city",
    "phone",
    "email"
).show()

+------------+---------+----------+---------------+
|        name|     city|     phone|          email|
+------------+---------+----------+---------------+
| Aarav Mehta|Hyderabad|9876500011| aarav@mail.com|
|   Sana Khan|Bangalore|      NULL|  sana@mail.com|
| John Mathew|     NULL|9876500013|           NULL|
|Ayesha Begum|Hyderabad|9876500014|ayesha@mail.com|
|  Vikram Rao|   Mumbai|      NULL|           NULL|
+------------+---------+----------+---------------+



In [40]:
flat_customers_df.filter(
    col("city").isNull()
).show()

+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|customer_id|       name|city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|          3|John Mathew|NULL|      Gold|9876500013| NULL|           Flight|        High|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+



In [41]:
flat_customers_df.filter(
    col("phone").isNull()
).show()

+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|      name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|          2| Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
|          5|Vikram Rao|   Mumbai|  Platinum| NULL|         NULL|           Flight|        High|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+



In [42]:
flat_customers_df.filter(
    col("email").isNull()
).show()

+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|customer_id|       name|  city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|          3|John Mathew|  NULL|      Gold|9876500013| NULL|           Flight|        High|
|          5| Vikram Rao|Mumbai|  Platinum|      NULL| NULL|           Flight|        High|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+



In [43]:
flat_customers_df.filter(
    col("membership").isNull()
).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [44]:
flat_customers_df.filter(
    col("preferred_service").isNull()
).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [45]:
flat_customers_df.filter(
    col("budget_range").isNull()
).show()

+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|     name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|          2|Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+



In [46]:
flat_customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in flat_customers_df.columns
]).show()

+-----------+----+----+----------+-----+-----+-----------------+------------+
|customer_id|name|city|membership|phone|email|preferred_service|budget_range|
+-----------+----+----+----------+-----+-----+-----------------+------------+
|          0|   0|   1|         1|    2|    2|                1|           1|
+-----------+----+----+----------+-----+-----+-----------------+------------+



In [47]:
clean_customers_df = flat_customers_df.na.fill({
    "city": "Unknown",
    "membership": "Standard",
    "phone": "Not Provided",
    "email": "Not Provided",
    "preferred_service": "Not Selected",
    "budget_range": "Unknown"
})

In [48]:
clean_customers_df.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [49]:
clean_customers_df = clean_customers_df.withColumn(
    "customer_quality_status",
    when(
        col("city").isNull() |
        col("phone").isNull() |
        col("email").isNull() |
        col("membership").isNull() |
        col("preferred_service").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)

clean_customers_df.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|               Complete|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|               Complete|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|               Complete|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|               Complete|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|

In [50]:
clean_customers_df.groupBy(
    "customer_quality_status"
).count().show()

+-----------------------+-----+
|customer_quality_status|count|
+-----------------------+-----+
|               Complete|    5|
+-----------------------+-----+



In [51]:
clean_customers_df.filter(
    col("customer_quality_status") == "Complete"
).show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|               Complete|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|               Complete|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|               Complete|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|               Complete|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|

In [52]:
clean_customers_df.filter(
    col("customer_quality_status") == "Incomplete"
).show()

+-----------+----+----+----------+-----+-----+-----------------+------------+-----------------------+
|customer_id|name|city|membership|phone|email|preferred_service|budget_range|customer_quality_status|
+-----------+----+----+----------+-----+-----+-----------------+------------+-----------------------+
+-----------+----+----+----------+-----+-----+-----------------+------------+-----------------------+



In [53]:
clean_customers_df.groupBy(
    "membership"
).count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum|    1|
|    Silver|    1|
|      Gold|    2|
|  Standard|    1|
+----------+-----+



In [54]:
clean_customers_df.groupBy(
    "preferred_service"
).count().show()

+-----------------+-----+
|preferred_service|count|
+-----------------+-----+
|     Not Selected|    1|
|            Hotel|    1|
|           Flight|    3|
+-----------------+-----+



In [55]:
flat_customers_df.write.mode(
    "overwrite"
).parquet("customers_flat.parquet")

In [56]:
clean_customers_df.write.mode(
    "overwrite"
).option(
    "header",
    "true"
).csv("clean_customers.csv")

In [57]:
print("Original Count :", flat_customers_df.count())
print("Clean Count    :", clean_customers_df.count())

Original Count : 5
Clean Count    : 5


In [58]:
flat_customers_df.filter(
    col("phone").isNull() |
    col("email").isNull()
).show()

+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|customer_id|       name|     city|membership|     phone|        email|preferred_service|budget_range|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|          2|  Sana Khan|Bangalore|    Silver|      NULL|sana@mail.com|            Hotel|        NULL|
|          3|John Mathew|     NULL|      Gold|9876500013|         NULL|           Flight|        High|
|          5| Vikram Rao|   Mumbai|  Platinum|      NULL|         NULL|           Flight|        High|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+



In [59]:
flat_customers_df.filter(
    col("preferred_service").isNull() |
    col("budget_range").isNull()
).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+

